# Nim OOD Probe: Legality and Strategy

**Two sub-probes for DeepSeek-R1-Distill-7B on Nim:**

**Probe 1 — Legality:** Show pile state without listing legal moves. Ask for a move.
Check if valid (1 ≤ N ≤ pile size). Nim legality is trivial arithmetic — expect high rates even at B=0.

**Probe 2 — Strategy/Memorization:** Show positions where nim_sum ≠ 0. Check if model picks nim-optimal move.
Canonical piles = memorization test. Non-canonical = genuine reasoning test.

Data: `data/probes/nim_ood.jsonl`


In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path('..') if Path('..').joinpath('src').exists() else Path('.')
DATA_FILE = ROOT / 'data' / 'probes' / 'nim_ood.jsonl'

records = []
if DATA_FILE.exists():
    with open(DATA_FILE) as f:
        for line in f:
            records.append(json.loads(line))
else:
    print(f'WARNING: {DATA_FILE} not found. Run scripts/probe_nim_ood.py first.')

df = pd.DataFrame(records)
print(f'Total records: {len(df)}')
if len(df) > 0:
    print(f'Probes: {df["probe"].value_counts().to_dict()}')
    print(f'Budgets: {sorted(df["budget"].unique())}')


## Probe 1: Legality Rate by Budget

Expected: HIGH (>80%) even at B=0 — unlike Reversi (0% at B=0).

In [ ]:
if len(df) == 0 or 'probe' not in df.columns:
    print('No data yet.')
else:
    leg = df[df['probe'] == 'legality'].copy()
    if len(leg) == 0:
        print('No legality records yet.')
    else:
        summ = leg.groupby('budget').agg(
            n=('is_legal', 'count'),
            n_legal=('is_legal', 'sum'),
            n_no_parse=('outcome', lambda x: (x == 'no_parse').sum()),
            n_illegal=('outcome', lambda x: (x == 'illegal').sum()),
        )
        summ['legal_pct'] = (summ['n_legal'] / summ['n'] * 100).round(1)
        print(summ.to_string())


In [ ]:
if len(df) > 0 and 'probe' in df.columns:
    leg = df[df['probe'] == 'legality']
    if len(leg) > 0:
        summ = leg.groupby('budget')['is_legal'].agg(['mean','count']).reset_index()
        summ.columns = ['budget', 'legal_rate', 'n']
        fig, ax = plt.subplots(figsize=(7, 4))
        bars = ax.bar(range(len(summ)), summ['legal_rate']*100,
                      color='steelblue', edgecolor='white', linewidth=0.5)
        ax.set_xticks(range(len(summ)))
        ax.set_xticklabels([f'B={b}' for b in summ['budget']])
        ax.set_ylabel('Legal move rate (%)')
        ax.set_title('Nim Legality Probe: Legal Rate vs CoT Budget')
        ax.set_ylim(0, 110)
        ax.axhline(100, color='green', linestyle='--', alpha=0.4, label='100% (perfect)')
        ax.axhline(0, color='red', linestyle=':', alpha=0.5, label='Reversi B=0 baseline (0%)')
        ax.legend()
        for i, (bar, val) in enumerate(zip(bars, summ['legal_rate']*100)):
            ax.text(i, val+1, f'{val:.0f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
        plt.tight_layout()
        plt.savefig(ROOT/'data'/'probes'/'nim_legality_rate.png', dpi=150, bbox_inches='tight')
        plt.show()


## Probe 2: Strategy/Memorization Rate

- **Canonical piles** ([3,5,7] etc.) — textbook; model may have memorised nim-sum
- **Non-canonical** ([4,6,11] etc.) — genuinely novel; requires computing nim XOR

Memorization signature: canonical rate ≫ non-canonical at B=0.
Reasoning signature: both rates rise with budget.

In [ ]:
if len(df) > 0 and 'probe' in df.columns:
    strat = df[df['probe'] == 'strategy'].copy()
    if len(strat) == 0:
        print('No strategy data yet.')
    else:
        summ = strat.groupby(['tag','budget']).agg(
            n=('is_optimal', 'count'), n_opt=('is_optimal', 'sum')
        ).reset_index()
        summ['optimal_rate'] = summ['n_opt'] / summ['n']
        print(summ.to_string(index=False))


In [ ]:
if len(df) > 0 and 'probe' in df.columns:
    strat = df[df['probe'] == 'strategy']
    if len(strat) > 0:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
        for ax, tag, color in zip(axes, ['canonical','non_canonical'], ['darkorange','purple']):
            sub = strat[strat['tag']==tag].groupby('budget')['is_optimal'].agg(['mean','count']).reset_index()
            sub.columns = ['budget','optimal_rate','n']
            if len(sub) == 0:
                ax.text(0.5, 0.5, f'No {tag} data yet', ha='center', va='center', transform=ax.transAxes)
                continue
            ax.plot(sub['budget'], sub['optimal_rate']*100, 'o-', color=color, linewidth=2, markersize=8)
            ax.set_xticks(sub['budget'])
            ax.set_xlabel('CoT Budget (tokens)')
            ax.set_ylabel('Nim-optimal play rate (%)')
            lbl = 'Canonical piles\n(memorization test)' if tag=='canonical' else 'Non-canonical piles\n(reasoning test)'
            ax.set_title(lbl)
            ax.set_ylim(-5, 110)
            ax.axhline(100, color='green', linestyle='--', alpha=0.4, label='100% optimal')
            ax.axhline(50, color='gray', linestyle=':', alpha=0.4, label='Random baseline (50%)')
            ax.legend(fontsize=8)
            for _, row in sub.iterrows():
                ax.annotate(f'{row["optimal_rate"]*100:.0f}%', (row['budget'], row['optimal_rate']*100+2),
                            ha='center', va='bottom', fontsize=9)
        plt.suptitle('Nim Strategy Probe: Nim-Optimal Play Rate', fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.savefig(ROOT/'data'/'probes'/'nim_strategy_rate.png', dpi=150, bbox_inches='tight')
        plt.show()


## Summary: Nim vs Reversi OOD Comparison

In [ ]:
rows = []
if len(df) > 0 and 'probe' in df.columns:
    leg = df[df['probe']=='legality']
    for B, grp in leg.groupby('budget'):
        rows.append({'Game':'Nim (Legality)', 'Budget':B, 'Rate':f'{grp["is_legal"].mean():.0%}', 'N':len(grp)})
# Reversi v2 probe results (hard-coded from previous experiment)
for B, rate, n in [(0,'0%',15),(64,'0%',15),(256,'0%',15),(512,'6.7%',15)]:
    rows.append({'Game':'Reversi (Legality)', 'Budget':B, 'Rate':rate, 'N':n})
if rows:
    summary_df = pd.DataFrame(rows)
    pivot = summary_df.pivot(index='Budget', columns='Game', values='Rate')
    print('Legal move rate without scaffolding:\n')
    print(pivot.to_string())
    print()
    print('Interpretation:')
    print('  Nim:    high → legality is NOT a bottleneck (arithmetic)')
    print('  Reversi: 0%  → spatial computation bottleneck; CoT controls strategy, not legality')
